In [4]:
import pandas as pd
import requests
import sys
import io
from io import  StringIO
import shutil
from pathlib import Path

url = "https://en.wikipedia.org/wiki/Leadership_approval_opinion_polling_for_the_next_United_Kingdom_general_election"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
}

response = requests.get(url, headers=headers)
response.raise_for_status()
        
html_content = StringIO(response.text)
        
tables = pd.read_html(html_content, flavor='lxml')

if not tables:
    print("No tables were found on the page.")



def clean_data(df):
    # The html table has a two line header which is not correct for dataframes
    # this seems to prevent normal operations on the data frame, so save it as a csv, reload ir and delete the first row
    df.to_csv('leaderstemp.csv', index=False)
    df1 = pd.read_csv("leaderstemp.csv")
    df1 = df1.tail(-1)


    # To remove 'note' rows change a col to numeric. Notes will be strings which convert to None
    # then drop the rows with None thus removing the notes
    df1['Sample size'] = pd.to_numeric(df1['Sample size'],errors='coerce')
    df1 = df1.dropna(subset=['Sample size']) # remove 'note' rows

    return df1


In [8]:
tables[18]

Date(s) conducted                Pollster Sample size             Labour  \
    Date(s) conducted                Pollster Sample size Unnamed: 3_level_1   
    Date(s) conducted                Pollster Sample size               Pos.   
0      17–21 Apr 2026              Ipsos[169]        2262                20%   
1      10–13 Apr 2026     More in Common[151]        2011                16%   
2      10–12 Apr 2026  Freshwater Strategy[4]        1250                25%   
3      20–24 Mar 2026              Ipsos[173]        2283                19%   
4      20–24 Mar 2026              Ipsos[174]        2518                27%   
..                ...                     ...         ...                ...   
116    20–22 Sep 2024             YouGov[274]        2132                32%   
117    24–27 Aug 2024     More In Common[275]        2015                25%   
118      5–6 Aug 2024             YouGov[276]        2163                39%   
119      5–8 Jul 2024        YouGov[277][246]        2102                47%   
120      5–6 Jul 2024         Ipsos[278][247]        1141                40%   

                                                Conservative  \
    Unnamed: 4_level_1 Unnamed: 5_level_1 Unnamed: 6_level_1   
                  Neg.                Net               Pos.   
0                  55%                –35                22%   
1                  59%                –43                18%   
2                  51%                –26                28%   
3                  54%                –35                20%   
4                  58%                –31                26%   
..                 ...                ...                ...   
116                59%                –27                24%   
117                45%                –20                15%   
118                53%                –14                23%   
119                46%                 +1                21%   
120                34%                 +6                20%   

                                                      Reform  ...  \
    Unnamed: 7_level_1 Unnamed: 8_level_1 Unnamed: 9_level_1  ...   
                  Neg.                Net               Pos.  ...   
0                  49%                –27                27%  ...   
1                  45%                –27                  –  ...   
2                  47%                –19                35%  ...   
3                  52%                –32                25%  ...   
4                  56%                –30                30%  ...   
..                 ...                ...                ...  ...   
116                67%                –43                26%  ...   
117                57%                –42                  –  ...   
118                70%                –47                  –  ...   
119                72%                –51                28%  ...   
120                59%                –39                25%  ...   

               Lib Dems               Green                      \
    Unnamed: 14_level_1 Unnamed: 15_level_1 Unnamed: 16_level_1   
                    Net                Pos.                Neg.   
0                   –15                 28%                 40%   
1                     –                   –                   –   
2                    +1                 34%                 33%   
3                   –14                 27%                 39%   
4                   –23                 28%                 47%   
..                  ...                 ...                 ...   
116                  –8                 40%                 42%   
117                   –                   –                   –   
118                   –                   –                   –   
119                  +8                 46%                 38%   
120                  +1                 33%                 28%   

                                       Your                      \
    Unnamed: 

In [9]:
df = clean_data(tables[18])
df

,Date(s) conducted,Pollster,Sample size,Labour,Labour.1,Labour.2,Conservative,Conservative.1,Conservative.2,Reform,...,Lib Dems.2,Green,Green.1,Green.2,Your,Your.1,Your.2,Restore,Restore.1,Restore.2
2,17–21 Apr 2026,Ipsos[169],2262.0,20%,55%,–35,22%,49%,–27,27%,...,–15,28%,40%,–12,19%,33%,–14,19%,38%,–19
3,10–13 Apr 2026,More in Common[151],2011.0,16%,59%,–43,18%,45%,–27,–,...,–,–,–,–,–,–,–,–,–,–
4,10–12 Apr 2026,Freshwater Strategy[4],1250.0,25%,51%,–26,28%,47%,–19,35%,...,+1,34%,33%,+1,17%,42%,–25,23%,21%,+2
5,20–24 Mar 2026,Ipsos[173],2283.0,19%,54%,–35,20%,52%,–32,25%,...,–14,27%,39%,–12,16%,32%,–16,16%,38%,–22
6,20–24 Mar 2026,Ipsos[174],2518.0,27%,58%,–31,26%,56%,–30,30%,...,–23,28%,47%,–19,–,–,–,–,–,–
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,20–22 Sep 2024,YouGov[274],2132.0,32%,59%,–27,24%,67%,–43,26%,...,–8,40%,42%,–2,NaN,NaN,NaN,NaN,NaN,NaN
119,24–27 Aug 2024,More In Common[275],2015.0,25%,45%,–20,15%,57%,–42,–,...,–,–,–,–,NaN,NaN,NaN,NaN,NaN,NaN
120,5–6 Aug 2024,YouGov[276],2163.0,39%,53%,–14,23%,70%,–47,–,...,–,–,–,–,NaN,NaN,NaN,NaN,NaN,NaN
121,5–8 Jul 2024,YouGov[277][246],2102.0,47%,46%,+1,21%,72%,–51,28%,...,+8,46%,38%,+8,NaN,NaN,NaN,NaN,NaN,NaN
